## Get API data into a variable

In [ ]:
import requests
import json
from os import getenv

api_key = getenv("CORESIGNAL_API_KEY")
api_endpoint = "https://api.coresignal.com/cdapi/v2/job_base/search/filter"

headers = {
    "apikey": api_key,
    "Content-Type": "application/json"
}
body = {
    "last_updated_gte": "2026-04-21 00:00:00",
    "title": "data engineer"
}

all_results = []
current_page = 1
next_page_after = None

while True:
    url = api_endpoint
    if next_page_after:
        url = f"{api_endpoint}?after={next_page_after}"

    response = requests.post(url, headers=headers, json=body)
    
    total_pages = response.headers.get('x-total-pages', 'unknown')
    total_results = response.headers.get('x-total-results', 'unknown')
    next_page_after = response.headers.get('x-next-page-after')
    
    print(f"Page {current_page}/{total_pages} | Total Results: {total_results}")
    
    try:
        page_results = response.json()
        all_results.extend(page_results)
        print(f"  Retrieved {len(page_results)} results from this page")
    except json.JSONDecodeError:
        print(f"  Error parsing JSON response: {response.text}")
        break
    
    if not next_page_after or current_page >= int(total_pages):
        break
    
    current_page += 1

print(f"\n--- Summary ---")
print(f"Total results collected: {len(all_results)}")
print(f"Sample results: {all_results[:5] if all_results else 'No results'}")

## Save data to local database

In [ ]:
# Save all_results to local PostgreSQL database table
import psycopg2

database_user = getenv("DB_USER")
database_password = getenv("DB_PASSWORD")

conn = psycopg2.connect(
    host="localhost",
    database="market_fit",
    user=database_user,
    password=database_password
)

cur = conn.cursor()

for result in all_results:
    cur.execute("INSERT INTO your_table (column1, column2) VALUES (%s, %s)", (result['field1'], result['field2']))

conn.commit()
cur.close()
conn.close()